# S2F Performance Analysis

Use this notebook to explore the prediction outputs produced by different S2F runs and compare their behaviour. Each run writes a `prediction.df` file inside `<installation_directory>/output/<alias>/`. Update the configuration cells below with the aliases you want to analyse.

## Imports and configuration

The snippet below reads `s2f.conf` to locate the shared output directory. Adjust `CONFIG_PATH` if you are running the notebook from a different location.

In [2]:
from pathlib import Path
import configparser

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 10)
pd.set_option('display.width', 120)

sns.set_theme(style='whitegrid')

# Backwards-compatibility for deprecated NumPy aliases used by legacy modules
if not hasattr(np, 'float'):
    np.float = float  # type: ignore[attr-defined]
if not hasattr(np, 'int'):
    np.int = int  # type: ignore[attr-defined]
if not hasattr(np, 'bool'):
    np.bool = bool  # type: ignore[attr-defined]

CONFIG_PATH = Path('../s2f.conf')
config = configparser.ConfigParser()
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f'Configuration file not found: {CONFIG_PATH.resolve()}. Update CONFIG_PATH to match your setup.')
config.read(CONFIG_PATH)

BASE_OUTPUT = Path(config['directories']['installation_directory']).expanduser() / 'output'
if not BASE_OUTPUT.exists():
    raise FileNotFoundError(f'Output directory not found: {BASE_OUTPUT}. Make sure the S2F runs have been executed.')

FILTERED_GOA_PATH = Path(config.get('databases', 'filtered_goa', fallback='filtered_goa')).expanduser()
if not FILTERED_GOA_PATH.exists():
    raise FileNotFoundError(f'Filtered GOA file not found: {FILTERED_GOA_PATH}. Update FILTERED_GOA_PATH below if your data lives elsewhere.')

DEFAULT_OBO_PATH = Path('../go.obo').resolve()
if not DEFAULT_OBO_PATH.exists():
    raise FileNotFoundError(f'GO ontology file not found: {DEFAULT_OBO_PATH}. Update DEFAULT_OBO_PATH to point at your go.obo file.')

BASE_OUTPUT

PosixPath('/media/marcelo_baez/HD_Disc1/.S2F/output')

## Discover available runs

This cell lists the aliases that currently have diffusion outputs. If the list is long, slice or filter it as needed.

In [3]:
AVAILABLE_RUNS = sorted(p.name for p in BASE_OUTPUT.iterdir() if p.is_dir())
print(f"{len(AVAILABLE_RUNS)} run(s) found under {BASE_OUTPUT}")
pd.DataFrame({'alias': AVAILABLE_RUNS})

1 run(s) found under /media/marcelo_baez/HD_Disc1/.S2F/output


,alias
0,cafa3_testset


## Select runs to compare

Edit `RUNS_TO_COMPARE` to focus on the runs you are interested in. By default the cell keeps only aliases that actually exist in `AVAILABLE_RUNS`.

In [4]:
RUNS_TO_COMPARE = [
    'cafa3_testset',
]

RUNS_TO_COMPARE = [alias for alias in RUNS_TO_COMPARE if alias in AVAILABLE_RUNS]
if not RUNS_TO_COMPARE:
    raise ValueError('Update RUNS_TO_COMPARE with at least one valid alias.')

RUNS_TO_COMPARE

['cafa3_testset']

## Load prediction tables

Helper functions to read the prediction scores and the index files written by each run.

In [5]:
def load_predictions(alias: str, base_path: Path = BASE_OUTPUT) -> pd.DataFrame:
    """Load the diffusion output for a single run as a tidy DataFrame."""
    path = base_path / alias / 'prediction.df'
    if not path.exists():
        raise FileNotFoundError(f'Prediction file not found for {alias}: {path}')
    df = pd.read_csv(path, sep='	', header=None, names=['protein_id', 'term_id', 'score'])
    df['alias'] = alias
    return df


def load_terms(alias: str, base_path: Path = BASE_OUTPUT) -> pd.DataFrame:
    """Grab the GO term lookup table if you need term names or namespaces."""
    path = base_path / alias / 'terms.df'
    if not path.exists():
        raise FileNotFoundError(f'GO term index not found for {alias}: {path}')
    return pd.read_pickle(path)


def load_proteins(alias: str, base_path: Path = BASE_OUTPUT) -> pd.DataFrame:
    """Fetch the protein index for convenience (e.g. to map back to FASTA identifiers)."""
    path = base_path / alias / 'proteins.df'
    if not path.exists():
        raise FileNotFoundError(f'Protein index not found for {alias}: {path}')
    return pd.read_pickle(path)


In [6]:
predictions_df = pd.concat(
    [load_predictions(alias) for alias in RUNS_TO_COMPARE],
    ignore_index=True
)

print(f"Loaded {predictions_df.shape[0]:,} scored annotations spanning {predictions_df['alias'].nunique()} run(s).")
predictions_df.head()

Loaded 81,312,000 scored annotations spanning 1 run(s).


,protein_id,term_id,score,alias
0,T100900000026,GO:0000001,2.531924e-06,cafa3_testset
1,T100900000026,GO:0000006,4.486653e-07,cafa3_testset
2,T100900000026,GO:0000007,4.098290e-07,cafa3_testset
3,T100900000026,GO:0000009,1.930866e-06,cafa3_testset
4,T100900000026,GO:0000011,4.024371e-06,cafa3_testset


## Focus on top-N predictions per protein

Restrict to the highest-scoring annotations per protein to study agreement between runs.

In [7]:
TOP_N = 5  # change to inspect more or fewer annotations per protein

_top_sorted = predictions_df.sort_values(
    ['alias', 'protein_id', 'score'],
    ascending=[True, True, False]
)

top_predictions = (
    _top_sorted
    .groupby(['alias', 'protein_id'], as_index=False)
    .head(TOP_N)
    .reset_index(drop=True)
)

top_predictions.head()

,protein_id,term_id,score,alias
0,T100900000026,GO:0008150,0.374900,cafa3_testset
1,T100900000026,GO:0065007,0.374312,cafa3_testset
2,T100900000026,GO:0050789,0.374285,cafa3_testset
3,T100900000026,GO:0050794,0.374269,cafa3_testset
4,T100900000026,GO:0003674,0.313409,cafa3_testset


In [30]:
# maximun score and min score
SCORE_RANGE = (predictions_df['score'].min(), predictions_df['score'].max())
print(f"Score range across all runs: {SCORE_RANGE[0]:.4f} to {SCORE_RANGE[1]:.4f}")

Score range across all runs: -0.0000 to 0.7657


In [32]:
# I want the row with the maximum score
max_score_row = predictions_df.loc[predictions_df['score'].idxmin()]
print("Row with maximum score:")
print(max_score_row)

Row with maximum score:
protein_id     T96060011059
term_id          GO:0004743
score             -0.000014
alias         cafa3_testset
Name: 63992138, dtype: object


## Ground truth configuration

Use the helpers below to discover metadata for the available runs and set up GOA-based evaluation parameters.

In [8]:
import re

def _parse_taxon_from_fasta(fasta_path):
    """Infer a taxon identifier from the FASTA file name."""
    stem = Path(fasta_path).stem
    match = re.search(r'(\d+)$', stem)
    if match:
        return match.group(1)
    digits = re.findall(r'\d+', stem)
    return digits[-1] if digits else None


def discover_run_metadata(run_config_paths):
    """Parse run configuration files to map aliases to useful metadata."""
    metadata = {}
    for conf_path in run_config_paths:
        parser = configparser.ConfigParser()
        parser.read(conf_path)
        if not parser.has_section('configuration'):
            continue
        alias = parser.get('configuration', 'alias', fallback='').strip()
        if not alias:
            continue
        entry = metadata.setdefault(alias, {})
        entry['run_config'] = conf_path
        if parser.has_option('configuration', 'fasta'):
            fasta_path = parser.get('configuration', 'fasta')
            entry['fasta'] = fasta_path
            taxon = _parse_taxon_from_fasta(fasta_path)
            if taxon:
                entry['taxon_id'] = taxon
    return metadata


# RUN_CONFIG_PATHS = sorted(Path('.').glob('run*.conf'))
RUN_CONFIG_PATHS = sorted(Path('.').glob('../cafa*.conf'))
RUN_METADATA = discover_run_metadata(RUN_CONFIG_PATHS)

metadata_table = pd.DataFrame(
    [
        {
            'alias': alias,
            'run_config': str(meta.get('run_config')),
            'fasta': meta.get('fasta'),
            'taxon_id': meta.get('taxon_id'),
        }
        for alias, meta in RUN_METADATA.items()
    ]
).sort_values('alias').reset_index(drop=True) if RUN_METADATA else pd.DataFrame(columns=['alias', 'run_config', 'fasta', 'taxon_id'])

metadata_table

,alias,run_config,fasta,taxon_id
0,cafa3_testset,../cafa3_testset.conf,/media/marcelo_baez/HD_Disc1/databases/Project...,None


In [9]:
DEFAULT_EVIDENCE_CODES = [
    code.strip()
    for code in config.get('options', 'evidence_codes', fallback='EXP,IDA,IPI,IMP,IGI,IEP,TAS,IC').split(',')
    if code.strip()
]


In [12]:

GROUND_TRUTH_CONFIG = {
    alias: {
        'taxon_id': meta.get('taxon_id'),
        'goa_path': FILTERED_GOA_PATH,
        'obo_path': DEFAULT_OBO_PATH,
        'evidence_codes': DEFAULT_EVIDENCE_CODES,
        'min_annotations_per_protein': 1,
        'term_frequency_range': (1, None),
    }
    for alias, meta in RUN_METADATA.items()
    if meta.get('taxon_id')
}

GROUND_TRUTH_CONFIG

{}

In [16]:
import sklearn
sklearn.__version__

'1.8.0'

In [17]:
import sys
import os
from pathlib import Path

try:
    HERE = Path(__file__).parent
except NameError:
    # __file__ is not defined in interactive environments (e.g. Jupyter); fall back to cwd
    HERE = Path.cwd()

# sys.path.append(str(HERE / "../"))

# sys.path.append(os.getcwd())
from Measures.measures import HX_py
from GOTool.GeneOntology import GeneOntology

_GOA_CACHE = {}



In [19]:

def load_goa_annotations_for_taxon(taxon_id, goa_path, evidence_codes, chunk_size=200_000):
    """Load GOA annotations for a single taxon from a potentially large GOA file."""
    goa_path = Path(goa_path)
    if not goa_path.exists():
        raise FileNotFoundError(f'GOA file not found: {goa_path}')
    cache_key = (goa_path.resolve(), str(taxon_id), tuple(sorted(evidence_codes)) if evidence_codes else ())
    if cache_key in _GOA_CACHE:
        return _GOA_CACHE[cache_key].copy()

    column_names = [
        'DB', 'DB Object ID', 'DB Object Symbol', 'Qualifier', 'GO ID',
        'DB Reference', 'Evidence Code', 'With', 'Aspect', 'DB Object Name',
        'Synonym', 'DB Object Type', 'Taxon', 'Date', 'Assigned By',
        'Annotation Extension', 'Gene Product Form ID'
    ]
    dtype = {name: str for name in column_names}
    pattern = fr'(?:^|\|)taxon:{taxon_id}(?:$|\|)'
    annotations = []

    for chunk in pd.read_csv(
        goa_path,
        sep='	',
        comment='!',
        header=None,
        names=column_names,
        dtype=dtype,
        chunksize=chunk_size,
        low_memory=False,
    ):
        mask = chunk['Taxon'].fillna('').str.contains(pattern, regex=True)
        if not mask.any():
            continue
        subset = chunk.loc[mask]
        if evidence_codes:
            subset = subset[subset['Evidence Code'].isin(evidence_codes)]
            if subset.empty:
                continue
        trimmed = subset[['DB Object ID', 'GO ID']].drop_duplicates()
        trimmed = trimmed.rename(columns={'DB Object ID': 'Protein'})
        trimmed['Score'] = 1.0
        annotations.append(trimmed)

    if annotations:
        result = pd.concat(annotations, ignore_index=True)
    else:
        result = pd.DataFrame(columns=['Protein', 'GO ID', 'Score'])

    _GOA_CACHE[cache_key] = result
    return result.copy()


def build_prediction_and_gold_matrices(predictions, annotations):
    if predictions.empty:
        raise ValueError('Prediction table is empty for the requested alias.')
    if annotations.empty:
        raise ValueError('Ground-truth annotations are empty for the requested taxon.')

    proteins = sorted(set(predictions['protein_id']) | set(annotations['Protein']))
    terms = sorted(set(predictions['term_id']) | set(annotations['GO ID']))

    protein_to_idx = {protein: idx for idx, protein in enumerate(proteins)}
    term_to_idx = {term: idx for idx, term in enumerate(terms)}

    prediction_matrix = np.zeros((len(proteins), len(terms)), dtype=np.float32)
    rows = predictions['protein_id'].map(protein_to_idx).to_numpy()
    cols = predictions['term_id'].map(term_to_idx).to_numpy()
    prediction_matrix[rows, cols] = predictions['score'].to_numpy(dtype=float)

    gold_matrix = np.zeros((len(proteins), len(terms)), dtype=np.float32)
    gt_rows = annotations['Protein'].map(protein_to_idx).to_numpy()
    gt_cols = annotations['GO ID'].map(term_to_idx).to_numpy()
    gold_matrix[gt_rows, gt_cols] = 1.0

    return prediction_matrix, gold_matrix, protein_to_idx, term_to_idx


def compute_information_content(term_to_idx, ontology, organism_name):
    ic = np.zeros(len(term_to_idx), dtype=np.float32)
    for term, idx in term_to_idx.items():
        try:
            ic[idx] = ontology.find_term(term).information_content(organism_name)
        except KeyError:
            ic[idx] = 0.0
    return ic


def filter_matrices(gold_matrix, prediction_matrix, ic_vector, min_annotations_per_protein=3, term_frequency_range=(1, 1_000_000)):
    lower, upper = term_frequency_range
    upper = float('inf') if upper is None else upper

    sumrow = gold_matrix.sum(axis=1)
    sumcol = gold_matrix.sum(axis=0)

    row_mask = sumrow >= min_annotations_per_protein
    col_mask = (sumcol >= lower) & (sumcol <= upper)

    filtered_pred = prediction_matrix[row_mask][:, col_mask]
    filtered_gold = gold_matrix[row_mask][:, col_mask]
    filtered_ic = ic_vector[col_mask]

    return filtered_pred, filtered_gold, filtered_ic, row_mask, col_mask


def evaluate_alias(alias, predictions_df, alias_cfg):
    alias_predictions = predictions_df[predictions_df['alias'] == alias].copy()
    if alias_predictions.empty:
        raise ValueError('No predictions found for this alias in predictions_df.')

    taxon_id = alias_cfg.get('taxon_id')
    if not taxon_id:
        raise ValueError('Taxon identifier is missing from GROUND_TRUTH_CONFIG.')

    goa_path = alias_cfg.get('goa_path', FILTERED_GOA_PATH)
    evidence_codes = alias_cfg.get('evidence_codes', DEFAULT_EVIDENCE_CODES)
    annotations = load_goa_annotations_for_taxon(taxon_id, goa_path, evidence_codes)
    if annotations.empty:
        raise ValueError('No annotations retrieved from GOA for the selected taxon and evidence codes.')

    organism_name = f'gt_{alias}'
    ontology_path = alias_cfg.get('obo_path', DEFAULT_OBO_PATH)
    ontology = GeneOntology(str(ontology_path), verbose=False)
    ontology.build_structure()
    ontology.load_annotations(annotations, organism_name)
    ontology.up_propagate_annotations(organism_name)
    propagated_annotations = ontology.get_annotations(organism_name)

    prediction_matrix, gold_matrix, protein_to_idx, term_to_idx = build_prediction_and_gold_matrices(alias_predictions, propagated_annotations)
    ic_vector = compute_information_content(term_to_idx, ontology, organism_name)

    freq_range = alias_cfg.get('term_frequency_range', (1, 1_000_000))
    min_ann = alias_cfg.get('min_annotations_per_protein', 3)
    filtered_pred, filtered_gold, filtered_ic, row_mask, col_mask = filter_matrices(
        gold_matrix,
        prediction_matrix,
        ic_vector,
        min_annotations_per_protein=min_ann,
        term_frequency_range=freq_range,
    )

    if filtered_gold.size == 0 or filtered_gold.sum() == 0:
        raise ValueError('No overlapping annotations left after filtering criteria were applied.')

    if np.unique(filtered_pred).size > 10000:
        filtered_pred = np.around(filtered_pred, decimals=4)

    measure = HX_py(filtered_pred, filtered_ic, organism_id=alias, verbose=False)
    results = {
        'overall': measure.compute_overall(filtered_gold),
        'per_gene': measure.compute_per_gene(filtered_gold),
        'per_term': measure.compute_per_term(filtered_gold),
    }
    context = {
        'proteins_considered': int(row_mask.sum()),
        'terms_considered': int(col_mask.sum()),
        'total_annotations': int(filtered_gold.sum()),
        'matrix_shape': filtered_gold.shape,
    }
    return results, context


def flatten_metrics(nested_metrics):
    flat = {}
    for block, metrics in nested_metrics.items():
        for key, value in metrics.items():
            if isinstance(value, (list, tuple, dict)):
                continue
            arr = np.asarray(value)
            if arr.ndim == 0:
                label = key if block != 'overall' else f'{block}::{key}'
                flat[label] = float(arr)
    return flat

In [25]:
def _positive_mask(series):
    numeric = pd.to_numeric(series, errors='coerce')
    mask = numeric.fillna(0).gt(0)
    if series.dtype == object:
        lowered = series.astype(str).str.strip().str.lower()
        mask = mask | lowered.isin({'1', 'true', 't', 'yes', 'y'})
    return mask


def parse_ground_truth_csvs(csv_paths, id_column=None):
    """Parse one or more binary protein x GO matrices into annotation format."""
    if not csv_paths:
        raise ValueError('csv_paths is empty. Provide at least one CSV file.')

    annotations = []
    for csv_path in csv_paths:
        csv_path = Path(csv_path)
        if not csv_path.exists():
            raise FileNotFoundError(f'Ground-truth CSV not found: {csv_path}')

        frame = pd.read_csv(csv_path)
        if frame.empty:
            continue

        protein_col = id_column or frame.columns[0]
        if protein_col not in frame.columns:
            raise ValueError(f"Protein ID column '{protein_col}' is missing in {csv_path}")

        term_cols = [col for col in frame.columns if col != protein_col]
        if not term_cols:
            continue

        melted = frame.melt(id_vars=[protein_col], value_vars=term_cols, var_name='GO ID', value_name='value')
        melted = melted[_positive_mask(melted['value'])]
        if melted.empty:
            continue

        trimmed = melted[[protein_col, 'GO ID']].rename(columns={protein_col: 'Protein'})
        trimmed['Score'] = 1.0
        annotations.append(trimmed)

    if not annotations:
        return pd.DataFrame(columns=['Protein', 'GO ID', 'Score'])

    return pd.concat(annotations, ignore_index=True).drop_duplicates()


def _propagate_annotations_if_requested(alias, annotations, alias_cfg):
    if not alias_cfg.get('propagate_ground_truth', False):
        return annotations

    ontology_path = alias_cfg.get('obo_path', DEFAULT_OBO_PATH)
    organism_name = f'gt_{alias}'
    ontology = GeneOntology(str(ontology_path), verbose=False)
    ontology.build_structure()
    ontology.load_annotations(annotations, organism_name)
    ontology.up_propagate_annotations(organism_name)
    return ontology.get_annotations(organism_name)


def evaluate_alias(alias, predictions_df, alias_cfg):
    alias_predictions = predictions_df[predictions_df['alias'] == alias].copy()
    if alias_predictions.empty:
        raise ValueError('No predictions found for this alias in predictions_df.')

    csv_paths = alias_cfg.get('ground_truth_csvs') or alias_cfg.get('gold_csv_paths')
    if csv_paths:
        id_column = alias_cfg.get('ground_truth_id_column') or alias_cfg.get('gold_id_column')
        annotations = parse_ground_truth_csvs(csv_paths, id_column=id_column)
        annotations = _propagate_annotations_if_requested(alias, annotations, alias_cfg)
        if annotations.empty:
            raise ValueError('No positive annotations were found in the configured ground-truth CSV files.')

        ontology_path = alias_cfg.get('obo_path', DEFAULT_OBO_PATH)
        ontology = GeneOntology(str(ontology_path), verbose=False)
        ontology.build_structure()
        ic_source = annotations if alias_cfg.get('propagate_ground_truth', False) else _propagate_annotations_if_requested(alias, annotations, {'propagate_ground_truth': True, 'obo_path': ontology_path})
        organism_name = f'gt_{alias}_ic'
        ontology.load_annotations(ic_source, organism_name)
        ontology.up_propagate_annotations(organism_name)

    else:
        taxon_id = alias_cfg.get('taxon_id')
        if not taxon_id:
            raise ValueError('Taxon identifier is missing from GROUND_TRUTH_CONFIG and no CSV ground truth was provided.')

        goa_path = alias_cfg.get('goa_path', FILTERED_GOA_PATH)
        evidence_codes = alias_cfg.get('evidence_codes', DEFAULT_EVIDENCE_CODES)
        annotations = load_goa_annotations_for_taxon(taxon_id, goa_path, evidence_codes)
        if annotations.empty:
            raise ValueError('No annotations retrieved from GOA for the selected taxon and evidence codes.')

        organism_name = f'gt_{alias}'
        ontology_path = alias_cfg.get('obo_path', DEFAULT_OBO_PATH)
        ontology = GeneOntology(str(ontology_path), verbose=False)
        ontology.build_structure()
        ontology.load_annotations(annotations, organism_name)
        ontology.up_propagate_annotations(organism_name)
        annotations = ontology.get_annotations(organism_name)

    prediction_matrix, gold_matrix, protein_to_idx, term_to_idx = build_prediction_and_gold_matrices(alias_predictions, annotations)
    ic_vector = compute_information_content(term_to_idx, ontology, organism_name)

    freq_range = alias_cfg.get('term_frequency_range', (1, 1_000_000))
    min_ann = alias_cfg.get('min_annotations_per_protein', 3)
    filtered_pred, filtered_gold, filtered_ic, row_mask, col_mask = filter_matrices(
        gold_matrix,
        prediction_matrix,
        ic_vector,
        min_annotations_per_protein=min_ann,
        term_frequency_range=freq_range,
    )

    if filtered_gold.size == 0 or filtered_gold.sum() == 0:
        raise ValueError('No overlapping annotations left after filtering criteria were applied.')

    if np.unique(filtered_pred).size > 10000:
        filtered_pred = np.around(filtered_pred, decimals=4)

    measure = HX_py(filtered_pred, filtered_ic, organism_id=alias, verbose=False)
    results = {
        'overall': measure.compute_overall(filtered_gold),
        'per_gene': measure.compute_per_gene(filtered_gold),
        'per_term': measure.compute_per_term(filtered_gold),
    }
    context = {
        'proteins_considered': int(row_mask.sum()),
        'terms_considered': int(col_mask.sum()),
        'total_annotations': int(filtered_gold.sum()),
        'matrix_shape': filtered_gold.shape,
        'ground_truth_source': 'csv' if csv_paths else 'goa',
    }
    return results, context


# Example configuration for CSV-based ground truth:
GROUND_TRUTH_CONFIG['cafa3_testset'] = {
    'ground_truth_csvs': [
        '/media/marcelo_baez/HD_Disc1/databases/ProjectData/cafa3/TEMPROT/test/bp-test.csv', 
        '/media/marcelo_baez/HD_Disc1/databases/ProjectData/cafa3/TEMPROT/test/mf-test.csv', 
        '/media/marcelo_baez/HD_Disc1/databases/ProjectData/cafa3/TEMPROT/test/cc-test.csv'
    ],
    'ground_truth_id_column': None,  # None -> first column is used as protein ID
    'obo_path': DEFAULT_OBO_PATH,
    'propagate_ground_truth': True,
    'min_annotations_per_protein': 3,
    'term_frequency_range': (1, None),
}



In [26]:
GROUND_TRUTH_CONFIG

{'cafa3_testset': {'ground_truth_csvs': ['/media/marcelo_baez/HD_Disc1/databases/ProjectData/cafa3/TEMPROT/test/bp-test.csv',
   '/media/marcelo_baez/HD_Disc1/databases/ProjectData/cafa3/TEMPROT/test/mf-test.csv',
   '/media/marcelo_baez/HD_Disc1/databases/ProjectData/cafa3/TEMPROT/test/cc-test.csv'],
  'ground_truth_id_column': None,
  'obo_path': PosixPath('/home/marcelo_baez/paccanaro-lab/S2F/go.obo'),
  'propagate_ground_truth': True,
  'min_annotations_per_protein': 3,
  'term_frequency_range': (1, None)}}

In [21]:
EVALUATION_RESULTS = {}
EVALUATION_CONTEXT = {}
EVALUATION_SUMMARY = []
EVALUATION_ERRORS = []

for alias in RUNS_TO_COMPARE:
    cfg = GROUND_TRUTH_CONFIG.get(alias)
    if cfg is None:
        print(f'Skipping {alias}: no ground truth configuration available.')
        continue
    try:
        metrics, context = evaluate_alias(alias, predictions_df, cfg)
    except Exception as exc:
        print(f'{alias}: {exc}')
        EVALUATION_ERRORS.append({'alias': alias, 'error': str(exc)})
        continue

    EVALUATION_RESULTS[alias] = metrics
    EVALUATION_CONTEXT[alias] = context

    flattened = flatten_metrics(metrics)
    row = {'alias': alias, **flattened}
    EVALUATION_SUMMARY.append(row)

if EVALUATION_SUMMARY:
    display(pd.DataFrame(EVALUATION_SUMMARY).set_index('alias'))
else:           
    print('No successful evaluations were recorded.')


,overall::AUC,overall::AUPR,overall::Precision at 0.2 Recall,overall::F_max,overall::NDCG,overall::Jaccard,overall::smin,AUC per-gene,AUPR per-gene,Precision at 0.2 Recall per-gene,F_max per-gene,NDCG per-gene,Jaccard per-gene,smin per-gene,AUC per-term,AUPR per-term,Precision at 0.2 Recall per-term,F_max per-term,NDCG per-term,Jaccard per-term,smin per-term
alias,,,,,,,,,,,,,,,,,,,,,
cafa3_testset,0.740038,0.153241,1.0,0.324211,0.810372,0.193468,194.018422,0.791288,0.254846,0.420152,0.466397,0.617845,0.325688,0.037868,0.798953,0.070286,0.174365,0.198816,0.333735,0.140634,0.639106


In [27]:
# Evaluation per GO domain (BP, MF, CC)

GO_DOMAIN_MAP = {
    'BP': 'biological_process',
    'MF': 'molecular_function',
    'CC': 'cellular_component',
}

_GO_DOMAIN_ALIASES = {
    'bp': 'biological_process',
    'biological_process': 'biological_process',
    'biological process': 'biological_process',
    'mf': 'molecular_function',
    'molecular_function': 'molecular_function',
    'molecular function': 'molecular_function',
    'cc': 'cellular_component',
    'cellular_component': 'cellular_component',
    'cellular component': 'cellular_component',
}



def normalize_go_domain(domain):
    key = str(domain).strip().lower()
    if key not in _GO_DOMAIN_ALIASES:
        valid = ', '.join(sorted(GO_DOMAIN_MAP.keys()))
        raise ValueError(f'Unknown GO domain: {domain}. Use one of: {valid}')
    return _GO_DOMAIN_ALIASES[key]



def _domain_label_from_namespace(namespace):
    for label, ns in GO_DOMAIN_MAP.items():
        if ns == namespace:
            return label
    return namespace



def resolve_ground_truth_csvs_by_domain(alias_cfg, domains=('BP', 'MF', 'CC')):
    """
    Resolve CSV paths per GO domain.

    Supported config layouts:
    1) Explicit mapping:
       ground_truth_csvs_by_domain = {'BP': 'bp.csv', 'MF': 'mf.csv', 'CC': 'cc.csv'}
    2) Ordered list:
       ground_truth_csvs = ['bp.csv', 'mf.csv', 'cc.csv']
       with optional ground_truth_csv_domain_order (defaults to BP, MF, CC)
    """
    domain_labels = [_domain_label_from_namespace(normalize_go_domain(d)) for d in domains]

    by_domain = alias_cfg.get('ground_truth_csvs_by_domain') or alias_cfg.get('gold_csv_paths_by_domain')
    if by_domain:
        resolved = {label: [] for label in domain_labels}
        for raw_domain, raw_paths in by_domain.items():
            label = _domain_label_from_namespace(normalize_go_domain(raw_domain))
            if isinstance(raw_paths, (str, Path)):
                paths = [raw_paths]
            else:
                paths = list(raw_paths)
            resolved[label] = [str(p) for p in paths]
        return resolved

    csv_paths = alias_cfg.get('ground_truth_csvs') or alias_cfg.get('gold_csv_paths')
    if not csv_paths:
        return {}

    if isinstance(csv_paths, (str, Path)):
        csv_paths = [csv_paths]

    domain_order = alias_cfg.get('ground_truth_csv_domain_order', domains)
    if len(csv_paths) != len(domain_order):
        raise ValueError(
            'ground_truth_csvs must have the same length as ground_truth_csv_domain_order '
            f'({len(csv_paths)} != {len(domain_order)}).'
        )

    resolved = {label: [] for label in domain_labels}
    for raw_domain, raw_paths in zip(domain_order, csv_paths):
        label = _domain_label_from_namespace(normalize_go_domain(raw_domain))
        if isinstance(raw_paths, (str, Path)):
            paths = [raw_paths]
        else:
            paths = list(raw_paths)
        resolved[label] = [str(p) for p in paths]
    return resolved



def _build_ontology(alias_cfg):
    ontology_path = alias_cfg.get('obo_path', DEFAULT_OBO_PATH)
    ontology = GeneOntology(str(ontology_path), verbose=False)
    ontology.build_structure()
    return ontology, ontology_path



def _domain_column_mask(term_to_idx, ontology, domain):
    namespace = normalize_go_domain(domain)
    mask = np.zeros(len(term_to_idx), dtype=bool)
    for term_id, idx in term_to_idx.items():
        try:
            term = ontology.find_term(term_id)
        except KeyError:
            continue
        mask[idx] = (term.domain == namespace)
    return mask, namespace



def _evaluate_domain_matrices(alias, domain_label, namespace, pred, gold, ic, alias_cfg, ground_truth_source):
    freq_range = alias_cfg.get('term_frequency_range', (1, 1_000_000))
    min_ann = alias_cfg.get('min_annotations_per_protein', 3)

    filtered_pred, filtered_gold, filtered_ic, row_mask, col_mask = filter_matrices(
        gold,
        pred,
        ic,
        min_annotations_per_protein=min_ann,
        term_frequency_range=freq_range,
    )

    if filtered_gold.size == 0 or filtered_gold.sum() == 0:
        return None, {
            'domain_namespace': namespace,
            'ground_truth_source': ground_truth_source,
            'proteins_before_filter': int(gold.shape[0]),
            'terms_before_filter': int(gold.shape[1]),
            'error': 'No annotations left after filtering criteria were applied.',
        }

    if np.unique(filtered_pred).size > 10000:
        filtered_pred = np.around(filtered_pred, decimals=4)

    measure = HX_py(filtered_pred, filtered_ic, organism_id=f'{alias}:{domain_label}', verbose=False)
    metrics = {
        'overall': measure.compute_overall(filtered_gold),
        'per_gene': measure.compute_per_gene(filtered_gold),
        'per_term': measure.compute_per_term(filtered_gold),
    }
    context = {
        'domain_namespace': namespace,
        'ground_truth_source': ground_truth_source,
        'proteins_considered': int(row_mask.sum()),
        'terms_considered': int(col_mask.sum()),
        'total_annotations': int(filtered_gold.sum()),
        'matrix_shape': filtered_gold.shape,
    }
    return metrics, context



def evaluate_alias_per_go_domain(alias, predictions_df, alias_cfg, domains=('BP', 'MF', 'CC')):
    """Evaluate one alias separately for BP/MF/CC, prioritizing per-domain CSV ground truth."""
    alias_predictions = predictions_df[predictions_df['alias'] == alias].copy()
    if alias_predictions.empty:
        raise ValueError('No predictions found for this alias in predictions_df.')

    ontology, ontology_path = _build_ontology(alias_cfg)
    domain_labels = [_domain_label_from_namespace(normalize_go_domain(d)) for d in domains]

    domain_results = {}
    domain_context = {}

    csv_by_domain = resolve_ground_truth_csvs_by_domain(alias_cfg, domains=domains)
    if csv_by_domain:
        id_column = alias_cfg.get('ground_truth_id_column') or alias_cfg.get('gold_id_column')

        for domain_label in domain_labels:
            namespace = GO_DOMAIN_MAP[domain_label]
            csv_paths = csv_by_domain.get(domain_label, [])
            if not csv_paths:
                domain_context[domain_label] = {
                    'domain_namespace': namespace,
                    'ground_truth_source': 'csv',
                    'error': 'No CSV path configured for this domain.',
                }
                continue

            annotations = parse_ground_truth_csvs(csv_paths, id_column=id_column)
            if annotations.empty:
                domain_context[domain_label] = {
                    'domain_namespace': namespace,
                    'ground_truth_source': 'csv',
                    'error': 'No positive annotations were found in this domain CSV.',
                }
                continue

            if alias_cfg.get('propagate_ground_truth', False):
                annotations = _propagate_annotations_if_requested(
                    f'{alias}_{domain_label}',
                    annotations,
                    {'propagate_ground_truth': True, 'obo_path': ontology_path},
                )

            if annotations.empty:
                domain_context[domain_label] = {
                    'domain_namespace': namespace,
                    'ground_truth_source': 'csv',
                    'error': 'Ground-truth annotations are empty after propagation.',
                }
                continue

            # Keep IC computation consistent with the global evaluation logic.
            if alias_cfg.get('propagate_ground_truth', False):
                ic_source = annotations
            else:
                ic_source = _propagate_annotations_if_requested(
                    f'{alias}_{domain_label}_ic_src',
                    annotations,
                    {'propagate_ground_truth': True, 'obo_path': ontology_path},
                )

            organism_name = f'gt_{alias}_{domain_label}_ic'
            ontology.load_annotations(ic_source, organism_name)
            ontology.up_propagate_annotations(organism_name)

            prediction_matrix, gold_matrix, _, term_to_idx = build_prediction_and_gold_matrices(alias_predictions, annotations)
            ic_vector = compute_information_content(term_to_idx, ontology, organism_name)

            # Safety check: keep only terms that belong to the expected domain.
            domain_mask, _ = _domain_column_mask(term_to_idx, ontology, domain_label)
            if domain_mask.any():
                prediction_matrix = prediction_matrix[:, domain_mask]
                gold_matrix = gold_matrix[:, domain_mask]
                ic_vector = ic_vector[domain_mask]

            metrics, context = _evaluate_domain_matrices(
                alias,
                domain_label,
                namespace,
                prediction_matrix,
                gold_matrix,
                ic_vector,
                alias_cfg,
                ground_truth_source='csv',
            )
            if metrics is not None:
                domain_results[domain_label] = metrics
            domain_context[domain_label] = context

        return domain_results, domain_context

    # Fallback when CSV-based ground truth is not configured.
    taxon_id = alias_cfg.get('taxon_id')
    if not taxon_id:
        raise ValueError('Taxon identifier is missing and no CSV ground truth was provided.')

    goa_path = alias_cfg.get('goa_path', FILTERED_GOA_PATH)
    evidence_codes = alias_cfg.get('evidence_codes', DEFAULT_EVIDENCE_CODES)
    annotations = load_goa_annotations_for_taxon(taxon_id, goa_path, evidence_codes)
    if annotations.empty:
        raise ValueError('No annotations retrieved from GOA for the selected taxon and evidence codes.')

    organism_name = f'gt_{alias}'
    ontology.load_annotations(annotations, organism_name)
    ontology.up_propagate_annotations(organism_name)
    annotations = ontology.get_annotations(organism_name)

    prediction_matrix, gold_matrix, _, term_to_idx = build_prediction_and_gold_matrices(alias_predictions, annotations)
    ic_vector = compute_information_content(term_to_idx, ontology, organism_name)

    for domain_label in domain_labels:
        domain_mask, namespace = _domain_column_mask(term_to_idx, ontology, domain_label)
        if not domain_mask.any():
            domain_context[domain_label] = {
                'domain_namespace': namespace,
                'ground_truth_source': 'goa',
                'error': 'No GO terms from this domain were found in the prediction/gold universe.',
            }
            continue

        domain_pred = prediction_matrix[:, domain_mask]
        domain_gold = gold_matrix[:, domain_mask]
        domain_ic = ic_vector[domain_mask]

        metrics, context = _evaluate_domain_matrices(
            alias,
            domain_label,
            namespace,
            domain_pred,
            domain_gold,
            domain_ic,
            alias_cfg,
            ground_truth_source='goa',
        )
        if metrics is not None:
            domain_results[domain_label] = metrics
        domain_context[domain_label] = context

    return domain_results, domain_context



def flatten_domain_metrics(domain_metrics):
    """Flatten nested metrics dict using DOMAIN::... prefixed keys."""
    flat = {}
    for domain_label, metrics in domain_metrics.items():
        for key, value in flatten_metrics(metrics).items():
            flat[f'{domain_label}::{key}'] = value
    return flat



def evaluate_runs_per_go_domain(runs_to_compare, predictions_df, ground_truth_config):
    """Batch evaluation helper for BP/MF/CC metrics across aliases."""
    results = {}
    context = {}
    summary_rows = []
    errors = []

    for alias in runs_to_compare:
        cfg = ground_truth_config.get(alias)
        if cfg is None:
            errors.append({'alias': alias, 'error': 'No ground truth configuration available.'})
            continue

        try:
            alias_results, alias_context = evaluate_alias_per_go_domain(alias, predictions_df, cfg)
        except Exception as exc:
            errors.append({'alias': alias, 'error': str(exc)})
            continue

        results[alias] = alias_results
        context[alias] = alias_context

        for domain_label, metrics in alias_results.items():
            flat = flatten_metrics(metrics)
            summary_rows.append({'alias': alias, 'domain': domain_label, **flat})

    summary_df = pd.DataFrame(summary_rows)
    if not summary_df.empty:
        summary_df = summary_df.set_index(['alias', 'domain']).sort_index()

    return results, context, summary_df, errors


# Example usage with three CSVs in BP/MF/CC order:
# GROUND_TRUTH_CONFIG['your_alias'].update({
#     'ground_truth_csv_domain_order': ('BP', 'MF', 'CC'),
# })
# DOMAIN_RESULTS, DOMAIN_CONTEXT, DOMAIN_SUMMARY_DF, DOMAIN_ERRORS = evaluate_runs_per_go_domain(
#     RUNS_TO_COMPARE,
#     predictions_df,
#     GROUND_TRUTH_CONFIG,
# )
# DOMAIN_SUMMARY_DF



In [29]:
# Domain comparison table (BP / MF / CC)
GROUND_TRUTH_CONFIG['cafa3_testset'].update({
    'ground_truth_csv_domain_order': ('BP', 'MF', 'CC'),
})

DOMAIN_RESULTS, DOMAIN_CONTEXT, DOMAIN_SUMMARY_DF, DOMAIN_ERRORS = evaluate_runs_per_go_domain(
    RUNS_TO_COMPARE,
    predictions_df,
    GROUND_TRUTH_CONFIG,
)

if DOMAIN_SUMMARY_DF.empty:
    raise ValueError('DOMAIN_SUMMARY_DF is empty. Check DOMAIN_ERRORS and your ground-truth CSV/domain configuration.')

TARGET_METRICS = ('AUC', 'AUPR', 'F_max', 'smin')


def _metric_selected(column_name, target_metrics=TARGET_METRICS):
    lower = column_name.lower()
    return any(metric.lower() in lower for metric in target_metrics)


# 1) Clean overall view: rows=alias, columns=(domain, metric)
overall_cols = [
    col for col in DOMAIN_SUMMARY_DF.columns
    if col.startswith('overall::') and _metric_selected(col)
]

overall_long = DOMAIN_SUMMARY_DF.reset_index()[['alias', 'domain', *overall_cols]].melt(
    id_vars=['alias', 'domain'],
    var_name='metric',
    value_name='value',
)
overall_long['metric'] = overall_long['metric'].str.replace('overall::', '', regex=False)

overall_comparison_df = overall_long.pivot_table(
    index='alias',
    columns=['domain', 'metric'],
    values='value',
)
overall_comparison_df = overall_comparison_df.sort_index(axis=1, level=[0, 1])

display(overall_comparison_df)

# 2) Optional detailed view with per-gene/per-term variants (same target metrics)
detail_cols = [col for col in DOMAIN_SUMMARY_DF.columns if _metric_selected(col)]
detail_df = DOMAIN_SUMMARY_DF[detail_cols].copy()

display(detail_df)

if 'DOMAIN_ERRORS' in globals() and DOMAIN_ERRORS:
    print('DOMAIN_ERRORS')
    display(pd.DataFrame(DOMAIN_ERRORS))



domain               BP                                        CC                                       MF            \
metric              AUC      AUPR     F_max        smin       AUC      AUPR     F_max       smin       AUC      AUPR   
alias                                                                                                                  
cafa3_testset  0.911729  0.370703  0.483363  149.410963  0.946911  0.532755  0.615484  39.144068  0.955756  0.460664   

domain                              
metric            F_max       smin  
alias                               
cafa3_testset  0.573259  38.117572

overall::AUC  overall::AUPR  overall::F_max  overall::smin  AUC per-gene  AUPR per-gene  \
alias         domain                                                                                            
cafa3_testset BP          0.911729       0.370703        0.483363     149.410963      0.964489       0.587550   
              CC          0.946911       0.532755        0.615484      39.144068      0.979425       0.617822   
              MF          0.955756       0.460664        0.573259      38.117572      0.974448       0.584821   

                      F_max per-gene  smin per-gene  AUC per-term  AUPR per-term  F_max per-term  smin per-term  
alias         domain                                                                                             
cafa3_testset BP            0.704364       0.041761      0.829126       0.084650        0.229044       0.190322  
              CC            0.833320       0.065605      0.879908       0.155502        0.361828       0.235451  
              MF            0.789747       0.045379      0.915345       0.252605        0.516044       0.122834

In [22]:
metrics_df = pd.DataFrame(EVALUATION_SUMMARY).set_index('alias') if EVALUATION_SUMMARY else pd.DataFrame()
if not metrics_df.empty:
    TARGET_METRICS = ('AUC', 'AUPR', 'F_max', 'smin')
    metrics_df = metrics_df[[
        col for col in metrics_df.columns
        if any(metric.lower() in col.lower() for metric in TARGET_METRICS)
    ]]
    metrics_df.sort_index(inplace=True)
metrics_df

,overall::AUC,overall::AUPR,overall::F_max,overall::smin,AUC per-gene,AUPR per-gene,F_max per-gene,smin per-gene,AUC per-term,AUPR per-term,F_max per-term,smin per-term
alias,,,,,,,,,,,,
cafa3_testset,0.740038,0.153241,0.324211,194.018422,0.791288,0.254846,0.466397,0.037868,0.798953,0.070286,0.198816,0.639106


Simulating scenario where the maximun score is 0.5

In [36]:
_simulation_predictions_df = predictions_df.copy()
condition = _simulation_predictions_df['score'] > 0.5
# delete some rows to simulate missing predictions
_simulation_predictions_df = _simulation_predictions_df[condition]
try:
    evaluate_alias('cafa3_testset', _simulation_predictions_df, GROUND_TRUTH_CONFIG['cafa3_testset'])
except ValueError as e:
    print(f'Expected error for missing predictions: {e}')

SIM_DOMAIN_RESULTS, SIM_DOMAIN_CONTEXT, SIM_DOMAIN_SUMMARY_DF, SIM_DOMAIN_ERRORS = evaluate_runs_per_go_domain(
    RUNS_TO_COMPARE,
    _simulation_predictions_df,
    GROUND_TRUTH_CONFIG,
)
TARGET_METRICS = ('AUC', 'AUPR', 'F_max', 'smin')


def _metric_selected(column_name, target_metrics=TARGET_METRICS):
    lower = column_name.lower()
    return any(metric.lower() in lower for metric in target_metrics)


# 1) Clean overall view: rows=alias, columns=(domain, metric)
overall_cols = [
    col for col in SIM_DOMAIN_SUMMARY_DF.columns
    if col.startswith('overall::') and _metric_selected(col)
]

overall_long = SIM_DOMAIN_SUMMARY_DF.reset_index()[['alias', 'domain', *overall_cols]].melt(
    id_vars=['alias', 'domain'],
    var_name='metric',
    value_name='value',
)
overall_long['metric'] = overall_long['metric'].str.replace('overall::', '', regex=False)

overall_comparison_df = overall_long.pivot_table(
    index='alias',
    columns=['domain', 'metric'],
    values='value',
)
overall_comparison_df = overall_comparison_df.sort_index(axis=1, level=[0, 1])

display(overall_comparison_df)

# 2) Optional detailed view with per-gene/per-term variants (same target metrics)
detail_cols = [col for col in SIM_DOMAIN_SUMMARY_DF.columns if _metric_selected(col)]
detail_df = SIM_DOMAIN_SUMMARY_DF[detail_cols].copy()

display(detail_df)

domain               BP                                        CC                                       MF            \
metric              AUC      AUPR     F_max        smin       AUC      AUPR     F_max       smin       AUC      AUPR   
alias                                                                                                                  
cafa3_testset  0.503581  0.294103  0.016723  203.328368  0.502116  0.355632  0.055686  59.512011  0.517741  0.386472   

domain                              
metric            F_max       smin  
alias                               
cafa3_testset  0.068067  54.145778

overall::AUC  overall::AUPR  overall::F_max  overall::smin  AUC per-gene  AUPR per-gene  \
alias         domain                                                                                            
cafa3_testset BP          0.503581       0.294103        0.016723     203.328368      0.503694       0.020859   
              CC          0.502116       0.355632        0.055686      59.512011      0.501707       0.034038   
              MF          0.517741       0.386472        0.068067      54.145778      0.524016       0.073025   

                      F_max per-gene  smin per-gene  AUC per-term  AUPR per-term  F_max per-term  smin per-term  
alias         domain                                                                                             
cafa3_testset BP            0.010441      13.013289      0.501512       0.016038        0.009481      12.996753  
              CC            0.004877      10.559889      0.500073       0.030800        0.015430      10.414675  
              MF            0.058786       9.430443      0.503805       0.040125        0.021696       9.659973

## Visualising evaluation metrics

The cells below chart overall, per-gene, and per-term metrics using grouped bar plots (one subplot per metric).

In [ ]:
if metrics_df.empty:
    raise ValueError('metrics_df is empty. Run the evaluation cell above first.')

metrics_df = metrics_df.copy()
metrics_df.index.name = 'alias'
metrics_df.sort_index(inplace=True)

aliases = metrics_df.index.tolist()
if not aliases:
    raise ValueError('No aliases found in metrics_df.')

palette = sns.color_palette('tab10', n_colors=len(aliases))
alias_colors = dict(zip(aliases, palette))

block_specs = {
    # 'Overall': lambda col: col.startswith('overall::'),
    'Per gene': lambda col: 'per-gene' in col,
    'Per term': lambda col: 'per-term' in col,
}

for block_title, selector in block_specs.items():
    block_columns = [col for col in metrics_df.columns if selector(col)]
    if not block_columns:
        print(f'No metrics found for {block_title}. Skipping.')
        continue

    block_data = metrics_df[block_columns]

    def _clean_metric_name(name: str) -> str:
        if '::' in name:
            name = name.split('::', 1)[1]
        name = name.replace(' per-gene', '').replace(' per-term', '')
        return name

    metric_names = [_clean_metric_name(col) for col in block_columns]
    n_metrics = len(block_columns)
    ncols = min(3, n_metrics)
    nrows = int(np.ceil(n_metrics / ncols))

    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(4.5 * ncols, 3.2 * nrows), squeeze=False)
    fig.suptitle(f'{block_title}', fontsize=14)

    bar_height = 0.45
    y_positions = np.arange(len(aliases))

    for idx, (col, metric_name) in enumerate(zip(block_columns, metric_names)):
        ax = axes[idx // ncols][idx % ncols]
        values = block_data[col].astype(float).reindex(aliases)
        colors = [alias_colors[a] for a in aliases]
        bars = ax.barh(y_positions, values, height=bar_height, color=colors, edgecolor='black', linewidth=0.5)

        ax.set_title(metric_name)
        ax.set_xlim(0, 1)  # keep plots on a 0-1 scale for comparability
        # ax.set_ylabel('Alias')
        # ax.set_xlabel(metric_name)
        ax.set_yticks(y_positions)
        ax.set_yticklabels(aliases)

        offset = 0.01
        text_max = ax.get_xlim()[1] - offset
        text_min = offset
        for rect, value in zip(bars, values):
            y = rect.get_y() + rect.get_height() / 2
            if value >= 0:
                x = min(value + offset, text_max)
                ax.text(x, y, f'{value:.3f}', va='center', ha='left', fontsize=8)
            else:
                x = max(value - offset, text_min)
                ax.text(x, y, f'{value:.3f}', va='center', ha='right', fontsize=8)

    total_axes = nrows * ncols
    for j in range(n_metrics, total_axes):
        fig.delaxes(axes[j // ncols][j % ncols])

    handles = [
        plt.Line2D([0], [0], marker='s', color='w', markerfacecolor=alias_colors[alias], # noqa
                   markeredgecolor='black', markersize=8, label=alias)
        for alias in aliases
    ]
    # fig.legend(handles=handles, loc='upper right', title='Alias', frameon=False)
    fig.tight_layout(rect=[0, 0, 1, 0.95]) # type: ignore

plt.show()
